# ScamShield TR - 01: Baseline Model

Bu notebook, Türkçe SMS mesajlarını **spam / normal** olarak sınıflandıran ilk (baseline) modelleri eğitir.
Sonuçlar ve grafikler ara rapordaki **Yöntem** ve **Mevcut İlerleme** bölümlerinde kullanılacak.

**Veri seti:** [Turkish SMS Collection](https://www.kaggle.com/datasets/onurkarasoy/turkish-sms-collection) — 4.751 mesaj (2.536 spam, 2.215 normal)
Karasoy, O., Ballı, S. *Spam SMS Detection for Turkish Language with Deep Text Analysis and Deep Learning Methods.* Arab J Sci Eng (2021). https://doi.org/10.1007/s13369-021-06187-1

**Nasıl çalıştırılır?** Menüden **Çalışma zamanı → Tümünü çalıştır** (Runtime → Run all). İkinci hücre CSV dosyasını yüklemeni isteyecek.

| Adım | Ne yapıyoruz? |
|---|---|
| 1 | Veriyi yükle ve incele |
| 2 | Metinleri temizle (ön işleme) |
| 3 | Eğitim / test olarak ayır |
| 4 | 3 klasik modeli eğit ve karşılaştır |
| 5 | En iyi modeli detaylı değerlendir |
| 6 | Hata analizi ve modelin önemsediği kelimeler |
| 7 | Gerçek dolandırıcılık mesajlarıyla dene |

In [ ]:
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score,
                             classification_report, confusion_matrix,
                             f1_score, precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42          # Sonuçların her çalıştırmada aynı çıkması için
CIKTI = "ciktilar"         # Grafikler ve tablolar buraya kaydedilir
os.makedirs(CIKTI, exist_ok=True)
pd.set_option("display.max_colwidth", 120)

## 1. Veriyi yükle
Kaggle'dan indirdiğin zip'in içindeki **TurkishSMSCollection.csv** dosyasını seç.

In [ ]:
DOSYA_ADI = "TurkishSMSCollection.csv"
ADAY_YOLLAR = [DOSYA_ADI, os.path.join("data", DOSYA_ADI), os.path.join("..", "data", DOSYA_ADI)]

if not any(os.path.exists(y) for y in ADAY_YOLLAR):
    try:
        from google.colab import files  # Sadece Colab'da çalışır
        print("TurkishSMSCollection.csv dosyasını seç:")
        files.upload()
    except ImportError:
        raise FileNotFoundError("TurkishSMSCollection.csv bulunamadı. Dosyayı notebook ile aynı klasöre ya da data/ klasörüne koy.")

YOL = next(y for y in ADAY_YOLLAR if os.path.exists(y))


def veriyi_oku(yol):
    """CSV'nin ayırıcısı ve karakter kodlaması belli olmadığı için olasılıkları sırayla dener."""
    for kodlama in ["utf-8", "utf-8-sig", "cp1254", "latin-1"]:
        for ayirici in [";", ",", "\t"]:
            try:
                d = pd.read_csv(yol, sep=ayirici, encoding=kodlama)
            except Exception:
                continue
            d.columns = [c.strip() for c in d.columns]
            if {"Message", "Group"} <= set(d.columns):
                print(f"Okundu → ayırıcı: {ayirici!r}, kodlama: {kodlama}")
                return d
    raise ValueError("CSV okunamadı. Sütun adları 'Message' ve 'Group' olmalı.")


ham = veriyi_oku(YOL)
print("Toplam satır:", len(ham))
ham.head()

**Etiketler:** `Group` sütununda **1 = Spam**, **2 = Normal**. Aşağıdaki tablo bunu `GroupText` sütunuyla karşılaştırarak doğrular.

In [ ]:
if "GroupText" in ham.columns:
    display(pd.crosstab(ham["Group"], ham["GroupText"].fillna("(boş)")).iloc[:, :6])

df = ham[["Message", "Group"]].dropna().copy()
df["Message"] = df["Message"].astype(str).str.strip()
df = df[df["Message"] != ""]
df["etiket"] = (pd.to_numeric(df["Group"], errors="coerce") == 1).astype(int)   # 1 = Spam, 0 = Normal

# Aynı mesajın birden fazla kez geçmesi, test setine eğitimden "kopya" sızdırır → tekrarları siliyoruz
onceki = len(df)
df = df.drop_duplicates(subset="Message").reset_index(drop=True)
print(f"Tekrarlanan mesaj silindi: {onceki - len(df)}  →  kalan: {len(df)}")
print(df["etiket"].map({1: "Spam", 0: "Normal"}).value_counts())

### Veri keşfi (EDA)

In [ ]:
df["uzunluk"] = df["Message"].str.len()
fig, eksen = plt.subplots(1, 2, figsize=(12, 4))

sayilar = df["etiket"].map({1: "Spam", 0: "Normal"}).value_counts()
eksen[0].bar(sayilar.index, sayilar.values, color=["#d9534f" if i == "Spam" else "#5b8def" for i in sayilar.index])
eksen[0].set_title("Sınıf dağılımı")
eksen[0].set_ylabel("Mesaj sayısı")
for i, v in enumerate(sayilar.values):
    eksen[0].text(i, v, str(v), ha="center", va="bottom")

for et, ad, renk in [(0, "Normal", "#5b8def"), (1, "Spam", "#d9534f")]:
    eksen[1].hist(df.loc[df["etiket"] == et, "uzunluk"].clip(upper=300), bins=40, alpha=0.6, label=ad, color=renk)
eksen[1].set_title("Mesaj uzunluğu (karakter)")
eksen[1].set_xlabel("Karakter sayısı (300+ kırpıldı)")
eksen[1].legend()

plt.tight_layout()
plt.savefig(f"{CIKTI}/01_veri_dagilimi.png", dpi=150)
plt.show()
print(df.groupby(df["etiket"].map({1: "Spam", 0: "Normal"}))["uzunluk"].describe().round(1))

## 2. Ön işleme
- Türkçe'ye uygun küçük harfe çevirme (`I → ı`, `İ → i`)
- Türkçe karakterleri sadeleştirme (`ş → s`, `ı → i` …). Veri setindeki mesajların bir kısmı Türkçe karakter kullanmadan yazılmış; böylece *"başvuru"* ile *"basvuru"* aynı kelime sayılır.
- Linkleri, telefon numaralarını, para tutarlarını ve diğer sayıları **özel etiketlerle** değiştirme. Model belirli bir linki ezberlemek yerine *"mesajda link var"* kalıbını öğrenir.

In [ ]:
TR_SADE = str.maketrans("çğıöşüâîû", "cgiosuaiu")
URL = re.compile(r"(https?://\S+|www\.\S+|\b[\w-]+(\.[\w-]+)*\.(com|net|org|tr|ly|xyz|info|co|me|io|biz|online|site|top|club)\b(/\S*)?)")
TEL = re.compile(r"(\+?90\s?|0)?5\d{2}\s?\d{3}\s?\d{2}\s?\d{2}")
PARA = re.compile(r"\d+([.,]\d+)*\s?(tl|lira|try|₺)(?!\w)")
SAYI = re.compile(r"\d+")


def temizle(metin):
    m = str(metin).replace("I", "ı").replace("İ", "i").lower().translate(TR_SADE)
    m = URL.sub(" linkx ", m)
    m = TEL.sub(" telefonx ", m)
    m = PARA.sub(" parax ", m)
    m = SAYI.sub(" sayix ", m)
    return re.sub(r"\s+", " ", m).strip()


df["temiz"] = df["Message"].apply(temizle)
df[["Message", "temiz"]].sample(5, random_state=1)

## 3. Eğitim / test ayrımı
%80 eğitim, %20 test. `stratify` ile her iki kısımda da spam/normal oranı korunur.

In [ ]:
X_egitim, X_test, y_egitim, y_test = train_test_split(
    df["temiz"], df["etiket"], test_size=0.2, stratify=df["etiket"], random_state=RANDOM_STATE)
print("Eğitim:", len(X_egitim), " Test:", len(X_test))

## 4. Modeller
Metin, **TF-IDF** ile sayılara çevrilir. İki tür özellik birlikte kullanılır:
- **Kelime n-gram (1–2):** "kampanya", "hemen tikla" gibi kelime ve ikili kelime grupları
- **Karakter n-gram (2–5):** Yazım hatalarına ve ek almış kelimelere karşı dayanıklılık (*"kazandiniz"*, *"kazandınız!!"*)

Bu özellikler üzerinde 3 klasik model karşılaştırılır. Her model için 5 katlı çapraz doğrulama (cross-validation) F1 skoru da hesaplanır; böylece sonucun tek bir şanslı ayrıma bağlı olmadığı görülür.

In [ ]:
def ozellikler():
    return FeatureUnion([
        ("kelime", TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("karakter", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True)),
    ])


MODELLER = {
    "Lojistik Regresyon": LogisticRegression(max_iter=2000, C=10, class_weight="balanced"),
    "Naive Bayes": ComplementNB(alpha=0.3),
    "Doğrusal SVM": LinearSVC(C=1, class_weight="balanced"),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
sonuclar, egitilmis = [], {}
for ad, model in MODELLER.items():
    hat = Pipeline([("ozellik", ozellikler()), ("model", model)])
    cv_f1 = cross_val_score(hat, X_egitim, y_egitim, cv=cv, scoring="f1")
    hat.fit(X_egitim, y_egitim)
    tahmin = hat.predict(X_test)
    egitilmis[ad] = hat
    sonuclar.append({
        "Model": ad,
        "Accuracy": accuracy_score(y_test, tahmin),
        "Precision": precision_score(y_test, tahmin),
        "Recall": recall_score(y_test, tahmin),
        "F1": f1_score(y_test, tahmin),
        "CV F1 (ort ± std)": f"{cv_f1.mean():.4f} ± {cv_f1.std():.4f}",
    })
    print(f"{ad} eğitildi")

tablo = pd.DataFrame(sonuclar).sort_values("F1", ascending=False).reset_index(drop=True)
tablo.to_csv(f"{CIKTI}/model_karsilastirma.csv", index=False)
tablo.style.format({c: "{:.4f}" for c in ["Accuracy", "Precision", "Recall", "F1"]}).hide(axis="index")

> **Not:** Bu problemde **Recall** (spam'lerin ne kadarını yakaladık) özellikle önemli: Kaçırılan bir dolandırıcılık mesajı, yanlışlıkla şüpheli denen normal bir mesajdan daha zararlıdır.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
olcutler = ["Accuracy", "Precision", "Recall", "F1"]
genislik = 0.25
for i, satir in tablo.iterrows():
    ax.bar(np.arange(len(olcutler)) + i * genislik, satir[olcutler].astype(float), genislik, label=satir["Model"])
ax.set_xticks(np.arange(len(olcutler)) + genislik)
ax.set_xticklabels(olcutler)
alt = float(tablo[olcutler].astype(float).values.min())
ax.set_ylim(max(0.0, alt - 0.05), 1.0)
ax.set_title("Model karşılaştırması (test seti)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{CIKTI}/02_model_karsilastirma.png", dpi=150)
plt.show()

## 5. En iyi modelin detaylı değerlendirmesi

In [ ]:
en_iyi_ad = tablo.loc[0, "Model"]
en_iyi = egitilmis[en_iyi_ad]
tahmin = en_iyi.predict(X_test)
print("En iyi model:", en_iyi_ad, "\n")
print(classification_report(y_test, tahmin, target_names=["Normal", "Spam"], digits=4))

fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, tahmin), display_labels=["Normal", "Spam"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title(f"Karışıklık matrisi — {en_iyi_ad}")
ax.set_xlabel("Modelin tahmini")
ax.set_ylabel("Gerçek etiket")
plt.tight_layout()
plt.savefig(f"{CIKTI}/03_karisiklik_matrisi.png", dpi=150)
plt.show()

## 6. Hata analizi
Modelin yanlış bildiği mesajlar. Bunlar rapordaki **"Karşılaşılan sorunlar"** bölümü için malzeme: *Model neden yanıldı?*

In [ ]:
test_df = df.loc[X_test.index, ["Message", "etiket"]].copy()
test_df["tahmin"] = tahmin
hatalar = test_df[test_df["etiket"] != test_df["tahmin"]].copy()
hatalar["durum"] = np.where(hatalar["etiket"] == 1, "Kaçırılan spam", "Yanlış alarm")
print(hatalar["durum"].value_counts().to_string(), "\n")
hatalar.to_csv(f"{CIKTI}/hatali_tahminler.csv", index=False)
hatalar[["durum", "Message"]].head(15)

### Modelin önemsediği kelimeler
Lojistik Regresyon her kelimeye bir ağırlık verir. Pozitif ağırlık → spam'e işaret, negatif → normale işaret. Bu, modelin nasıl karar verdiğini açıklamanın basit bir yoludur (ileride LLM açıklama katmanına da girdi olacak).

In [ ]:
lr = egitilmis["Lojistik Regresyon"]
adlar = lr.named_steps["ozellik"].get_feature_names_out()
agirlik = lr.named_steps["model"].coef_[0]
kelime = np.array([a.startswith("kelime__") for a in adlar])
kelimeler = pd.Series(agirlik[kelime], index=[a.replace("kelime__", "") for a in adlar[kelime]])
spam_kelimeler = kelimeler.sort_values(ascending=False).head(15)
normal_kelimeler = kelimeler.sort_values().head(15)

fig, eksen = plt.subplots(1, 2, figsize=(12, 5))
eksen[0].barh(spam_kelimeler.index[::-1], spam_kelimeler.values[::-1], color="#d9534f")
eksen[0].set_title("Spam'e en çok işaret eden kelimeler")
eksen[1].barh(normal_kelimeler.index[::-1], -normal_kelimeler.values[::-1], color="#5b8def")
eksen[1].set_title("Normale en çok işaret eden kelimeler")
plt.tight_layout()
plt.savefig(f"{CIKTI}/04_onemli_kelimeler.png", dpi=150)
plt.show()

## 7. Gerçek dolandırıcılık mesajlarıyla deneme
Veri setindeki "spam" mesajların çoğu **reklam** mesajı. Bizim asıl hedefimiz ise **dolandırıcılık** (sahte kargo, banka, icra…). Aşağıda modelin bu tür mesajlara ne dediğine bakıyoruz.

Kendi örneklerini de listeye ekleyebilirsin.

In [ ]:
DENEME = [
    "Kargonuz adres yetersizligi nedeniyle teslim edilemedi. Adresinizi guncellemek icin: ptt-kargo-teslim.xyz",
    "Sayın müşterimiz hesabınız güvenlik nedeniyle bloke edilmiştir. Hemen giriş yapın: bit.ly/3xYzQ",
    "Hakkınızda icra takibi başlatılmıştır. Dosyanızı görüntülemek için tıklayınız: e-devlet-icra.online",
    "Tebrikler! 5000 TL değerinde hediye çeki kazandınız, hemen tıklayın",
    "Akşam yemeğe geliyor musun?",
    "Toplantı saat 3'e ertelendi, haber vereyim dedim",
    "Annem seni aradı, müsait olunca dönüş yap",
]
olasilik = lr.predict_proba([temizle(m) for m in DENEME])[:, 1]
deneme_df = pd.DataFrame({"Mesaj": DENEME, "Spam olasılığı": olasilik.round(3)})
deneme_df["Karar"] = np.where(deneme_df["Spam olasılığı"] >= 0.5, "Şüpheli", "Normal")
deneme_df.to_csv(f"{CIKTI}/deneme_mesajlari.csv", index=False)
deneme_df

## 8. Sonuçları indir
Tüm grafikler ve tablolar `ciktilar/` klasöründe. Aşağıdaki hücre hepsini tek bir zip olarak indirir.

In [ ]:
import shutil

shutil.make_archive("baseline_sonuclari", "zip", CIKTI)
try:
    from google.colab import files
    files.download("baseline_sonuclari.zip")
except ImportError:
    print("Sonuçlar: baseline_sonuclari.zip")

## Rapora ne koyulacak?

| Rapordaki bölüm | Bu notebook'tan |
|---|---|
| **Yöntem** | Ön işleme adımları (bölüm 2), TF-IDF kelime + karakter n-gram, 3 model, %80/%20 ayrım, 5 katlı çapraz doğrulama |
| **Mevcut ilerleme** | `model_karsilastirma.csv` tablosu, `01_veri_dagilimi.png`, `02_model_karsilastirma.png`, `03_karisiklik_matrisi.png`, `04_onemli_kelimeler.png` |
| **Karşılaşılan sorunlar** | Tekrarlanan mesajlar (silindi) · Türkçe karakterli / karaktersiz yazım farkı (sadeleştirme ile çözüldü) · Veri setindeki spam'lerin çoğu reklam, dolandırıcılık değil (→ kendi dolandırıcılık verimizi toplayacağız) · Hata analizindeki örnekler |
| **Sonraki adım** | BERTurk ile derin öğrenme modeli, dolandırıcılık türü sınıflandırması, kendi veri setimiz |